In [14]:

import os
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gc

data_dir = "/content/BreaKHis_AllMag_Split"
# HIZLI AYARLAR
batch_size           = 512          # Küçük batch = çok daha hızlı!
accumulation_steps   = 1           # Effective batch = 256
num_epochs_frozen    = 10           # Daha az epoch
num_epochs_finetune  = 15          # Daha az epoch
num_classes          = 8
lr_frozen            = 1e-3        # Daha yüksek LR = daha hızlı öğrenme
lr_finetune          = 1e-4
gamma                = 2.0         # Sabit gamma

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print("Device:", device)
print(f"Batch: {batch_size} × {accumulation_steps} = {batch_size * accumulation_steps} (effective)")
print(f"Epochs: {num_epochs_frozen + num_epochs_finetune} total")

Device: cuda
Batch: 512 × 1 = 512 (effective)
Epochs: 25 total


In [15]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

In [16]:

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),  # 45'ten 20'ye düşürüldü
    transforms.ColorJitter(brightness=0.15, contrast=0.15),  # Daha az
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Transform hazır")

Transform hazır


In [17]:

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

# Daha hızlı dataloader ayarları
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=3,
    drop_last=True  # Son küçük batch'i atla
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size * 2,  # Validation için daha büyük batch
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

class_names = train_dataset.classes
print("Classes:", class_names)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

Classes: ['adenosis', 'ductal_carcinoma', 'fibroadenoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma', 'phyllodes_tumor', 'tubular_adenoma']
Train: 5538 | Val: 1182 | Test: 1194
Batches per epoch: 10


In [18]:

model = models.densenet121(weights='IMAGENET1K_V1')

num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes)
)

# Tüm layers'ı dondur
for param in model.parameters():
    param.requires_grad = False

# Sadece classifier'ı aç
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: DenseNet121")
print(f"Total params: {total_params:,}")
print(f"Trainable: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")


Model: DenseNet121
Total params: 7,482,760
Trainable: 528,904 (7.1%)


In [19]:

def train_model_fast(model, optimizer, criterion, num_epochs, phase_name):

    scaler = torch.amp.GradScaler('cuda')
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[pg['lr'] * 10 for pg in optimizer.param_groups] if len(optimizer.param_groups) > 1 else optimizer.param_groups[0]['lr'] * 10,
        epochs=num_epochs,
        steps_per_epoch=len(train_loader) // accumulation_steps,
        pct_start=0.3
    )

    best_val_acc = 0.0
    best_model_wts = None

    for epoch in range(num_epochs):
        # TRAIN
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f"[{phase_name}] Epoch {epoch+1}/{num_epochs}")

        for batch_idx, (images, labels) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels) / accumulation_steps

            scaler.scale(loss).backward()

            if (batch_idx + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            running_loss += loss.item() * accumulation_steps * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Progress bar güncelle
            pbar.set_postfix({
                'loss': f"{running_loss/total:.4f}",
                'acc': f"{100*correct/total:.1f}%"
            })

        train_acc = 100 * correct / total
        train_loss = running_loss / total

        # VALIDATION
        model.eval()
        val_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_loss = val_loss / total

        print(f"\n[{phase_name}] Epoch {epoch+1}/{num_epochs}")
        print(f"  Train → Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
        print(f"  Val   → Loss: {val_loss:.4f} | Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = model.state_dict().copy()
            print(f"  ✅ NEW BEST! Val Acc: {val_acc:.2f}%")

        # Memory temizliği
        if epoch % 2 == 0:
            torch.cuda.empty_cache()

    if best_model_wts is not None:
        model.load_state_dict(best_model_wts)

    return model, best_val_acc

print("✅ Eğitim fonksiyonu hazır")

✅ Eğitim fonksiyonu hazır


In [20]:
#
# PHASE 1: FROZEN TRAINING
#
print("\n" + "="*50)
print("🔒 PHASE 1: FROZEN TRAINING")
print("="*50)

criterion = FocalLoss(alpha=None, gamma=gamma)
optimizer = optim.AdamW(model.classifier.parameters(), lr=lr_frozen, weight_decay=0.01)

model, best_frozen_acc = train_model_fast(
    model, optimizer, criterion, num_epochs_frozen, "FROZEN"
)

# Kaydet
save_path = "/content/drive/MyDrive/fast_frozen_model.pth"
torch.save(model.state_dict(), save_path)
print(f"\n💾 Frozen model kaydedildi → {save_path}")
print(f"🎯 Best Frozen Acc: {best_frozen_acc:.2f}%")

# Memory temizle
torch.cuda.empty_cache()
gc.collect()


🔒 PHASE 1: FROZEN TRAINING


[FROZEN] Epoch 1/10: 100%|██████████| 10/10 [01:21<00:00,  8.12s/it, loss=1.2887, acc=43.1%]



[FROZEN] Epoch 1/10
  Train → Loss: 1.2887 | Acc: 43.12%
  Val   → Loss: 1.0666 | Acc: 47.29%
  ✅ NEW BEST! Val Acc: 47.29%


[FROZEN] Epoch 2/10: 100%|██████████| 10/10 [01:21<00:00,  8.18s/it, loss=0.9968, acc=49.3%]



[FROZEN] Epoch 2/10
  Train → Loss: 0.9968 | Acc: 49.26%
  Val   → Loss: 0.8591 | Acc: 55.08%
  ✅ NEW BEST! Val Acc: 55.08%


...

[FROZEN] Epoch 9/10: 100%|██████████| 10/10 [01:13<00:00,  7.38s/it, loss=0.5723, acc=62.6%]



[FROZEN] Epoch 9/10
  Train → Loss: 0.5723 | Acc: 62.58%
  Val   → Loss: 0.4652 | Acc: 68.19%
  ✅ NEW BEST! Val Acc: 68.19%


[FROZEN] Epoch 10/10: 100%|██████████| 10/10 [01:13<00:00,  7.35s/it, loss=0.5688, acc=63.8%]



[FROZEN] Epoch 10/10
  Train → Loss: 0.5688 | Acc: 63.77%
  Val   → Loss: 0.4637 | Acc: 68.44%
  ✅ NEW BEST! Val Acc: 68.44%

💾 Frozen model kaydedildi → /content/drive/MyDrive/fast_frozen_model.pth
🎯 Best Frozen Acc: 68.44%


17

In [21]:
#
# PHASE 2: FINE-TUNING
#
print("\n" + "="*50)
print("🔓 PHASE 2: FINE-TUNING")
print("="*50)

# Sadece son iki denseblock + classifier'ı aç
for name, param in model.named_parameters():
    if any(block in name for block in ["denseblock3", "denseblock4", "classifier"]):
        param.requires_grad = True

# Farklı learning rate'ler
param_groups = [
    {'params': [p for n, p in model.named_parameters() if "denseblock3" in n and p.requires_grad],
     'lr': lr_finetune * 0.1},
    {'params': [p for n, p in model.named_parameters() if "denseblock4" in n and p.requires_grad],
     'lr': lr_finetune},
    {'params': [p for n, p in model.named_parameters() if "classifier" in n and p.requires_grad],
     'lr': lr_finetune * 3},
]

optimizer = optim.AdamW(param_groups, weight_decay=0.01)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"🔧 Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)\n")

model, best_finetune_acc = train_model_fast(
    model, optimizer, criterion, num_epochs_finetune, "FINETUNE"
)

# Kaydet
save_path = "/content/drive/MyDrive/fast_finetuned_model.pth"
torch.save(model.state_dict(), save_path)
print(f"\n💾 Fine-tuned model kaydedildi → {save_path}")
print(f"🎯 Best Fine-tune Acc: {best_finetune_acc:.2f}%")

torch.cuda.empty_cache()
gc.collect()


🔓 PHASE 2: FINE-TUNING
🔧 Trainable: 5,524,744 / 7,482,760 (73.8%)



[FINETUNE] Epoch 1/15: 100%|██████████| 10/10 [01:15<00:00,  7.55s/it, loss=0.5417, acc=65.0%]



[FINETUNE] Epoch 1/15
  Train → Loss: 0.5417 | Acc: 64.96%
  Val   → Loss: 0.4222 | Acc: 68.95%
  ✅ NEW BEST! Val Acc: 68.95%


[FINETUNE] Epoch 2/15: 100%|██████████| 10/10 [01:16<00:00,  7.61s/it, loss=0.4797, acc=66.7%]



[FINETUNE] Epoch 2/15
  Train → Loss: 0.4797 | Acc: 66.70%
  Val   → Loss: 0.3587 | Acc: 72.50%
  ✅ NEW BEST! Val Acc: 72.50%


...

[FINETUNE] Epoch 14/15: 100%|██████████| 10/10 [01:14<00:00,  7.48s/it, loss=0.0428, acc=94.5%]



[FINETUNE] Epoch 14/15
  Train → Loss: 0.0428 | Acc: 94.49%
  Val   → Loss: 0.1440 | Acc: 87.65%


[FINETUNE] Epoch 15/15: 100%|██████████| 10/10 [01:15<00:00,  7.55s/it, loss=0.0423, acc=94.3%]



[FINETUNE] Epoch 15/15
  Train → Loss: 0.0423 | Acc: 94.28%
  Val   → Loss: 0.1437 | Acc: 87.82%
  ✅ NEW BEST! Val Acc: 87.82%

💾 Fine-tuned model kaydedildi → /content/drive/MyDrive/fast_finetuned_model.pth
🎯 Best Fine-tune Acc: 87.82%


9

In [23]:
#
# TEST EVALUATION
#
print("\n" + "="*50)
print("🧪 TEST EVALUATION")
print("="*50)

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            outputs = model(images)

        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Sonuçlar
test_acc = 100 * (np.array(all_preds) == np.array(all_labels)).sum() / len(all_labels)
test_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"\n Test Accuracy: {test_acc:.2f}%")
print(f"Test F1-Score: {test_f1:.4f}\n")

print("Classification Report:")
print("="*60)
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

print("\nConfusion Matrix:")
print("="*60)
cm = confusion_matrix(all_labels, all_preds)
print(cm)

print("\nEĞİTİM TAMAMLANDI!")


🧪 TEST EVALUATION


Testing: 100%|██████████| 2/2 [00:20<00:00, 10.49s/it]



 Test Accuracy: 87.10%
Test F1-Score: 0.8689

Classification Report:
                     precision    recall  f1-score   support

           adenosis      0.942     0.956     0.949        68
   ductal_carcinoma      0.871     0.923     0.896       519
       fibroadenoma      0.877     0.889     0.883       153
  lobular_carcinoma      0.688     0.579     0.629        95
 mucinous_carcinoma      0.904     0.783     0.839       120
papillary_carcinoma      0.947     0.845     0.893        84
    phyllodes_tumor      0.814     0.826     0.820        69
    tubular_adenoma      0.912     0.965     0.938        86

           accuracy                          0.871      1194
          macro avg      0.869     0.846     0.856      1194
       weighted avg      0.870     0.871     0.869      1194


Confusion Matrix:
[[ 65   0   1   0   1   0   0   1]
 [  0 479   5  21   7   4   2   1]
 [  2   3 136   0   2   0  10   0]
 [  0  39   1  55   0   0   0   0]
 [  0  18   0   3  94   0   0   5]
 

In [24]:
#
# ÖZET RAPOR
#
print("\n" + "="*60)
print("EĞİTİM ÖZETİ")
print("="*60)
print(f"Frozen Epochs: {num_epochs_frozen} | Best Acc: {best_frozen_acc:.2f}%")
print(f"Finetune Epochs: {num_epochs_finetune} | Best Acc: {best_finetune_acc:.2f}%")
print(f"Final Test Acc: {test_acc:.2f}%")
print(f"Test F1-Score: {test_f1:.4f}")
print(f"Total Epochs: {num_epochs_frozen + num_epochs_finetune}")
print(f"Effective Batch Size: {batch_size * accumulation_steps}")
print("="*60)


EĞİTİM ÖZETİ
Frozen Epochs: 10 | Best Acc: 68.44%
Finetune Epochs: 15 | Best Acc: 87.82%
Final Test Acc: 87.10%
Test F1-Score: 0.8689
Total Epochs: 25
Effective Batch Size: 512
